# Affective Priming (J2) — Vector Injection vs. Real Prompt Simulation

Full pipeline, consolidating everything decided in this project:

**Phase 1**: extract a gratitude↔routine direction from the real Mega Study `g_writing` data (meandiff + probe, `extraction_tokens='all'`).

**Phase 2**: two arms, run across the same set of personas (`persona_summary`, live from the Twin-2K-500 dataset):
- **Simulation arm**: real system instruction + persona + real writing-task instruction (QID46 prime / QID47 control) + real downstream questions (manip check / Elevation / DV1), ONE combined generation producing Q1(writing)+Q2+Q3+Q4 — this replicates the actual J2 digital-twin methodology (no injection).
- **Injection arm**: same persona, but Q1 (writing) is skipped entirely; the gratitude↔routine vector is injected at every generation step (positive β = toward gratitude, negative β = toward routine, same vector, sign flipped) while generating Q2+Q3+Q4 only.

JSON output is constrained via `outlines` (schema-guaranteed), since an 8B model is much less reliable than GPT-4.1 at free-form JSON compliance.

**Caveat**: this notebook has not been run end-to-end on a live GPU session by me (no Colab/GPU access on my end) — it's built directly from the source code and real materials verified earlier in this project, but expect to hit a few bugs on first run. Paste back any errors and they can be fixed.

## 0. Install dependencies

In [ ]:
!git clone https://github.com/aminbana/PsySET.git
%cd PsySET
!pip install -q transformers==4.56.1 transformer_lens==2.16.1
!pip install -q pandas scikit-learn tqdm huggingface_hub requests anthropic

In [ ]:
!pip install -q anthropic

## 1. HuggingFace login
Add your token as a Colab secret named `HUGGINGFACE_TOKEN` first.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HUGGINGFACE_TOKEN')
login(token=HF_TOKEN)

## 2. Load Llama-3.1-8B-Instruct

In [ ]:
import sys
import os
import torch

PSYSET_DIR = "/content/PsySET"
if PSYSET_DIR not in sys.path:
    sys.path.append(PSYSET_DIR)
    sys.path.append(os.path.join(PSYSET_DIR, "LLMs"))
os.chdir(PSYSET_DIR)

from my_llama import SteeredLlamaForCausalLM
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
ASSISTANT_TAG = "<|start_header_id|>assistant<|end_header_id|>\n\n"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = SteeredLlamaForCausalLM.from_pretrained(
    MODEL_NAME, device_map="cuda", torch_dtype=torch.bfloat16
)
model.eval()
print("model loaded:", model.config.model_type, model.config.num_hidden_layers, "layers")

/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violations in the next r

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

... (5 outputs left out)

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

model loaded: llama 32 layers


# PHASE 1: Extract the gratitude↔routine vector
## 3. Download the real Mega Study writing data

In [ ]:
import pandas as pd

RESPONSE_CSV_URL = (
    "https://raw.githubusercontent.com/TianyiPeng/Twin-2K-500-Mega-Study/main/"
    ".dat/affective_priming/raw_data/response.csv"
)
RESPONSE_CSV_PATH = "response.csv"
!curl -sL "{RESPONSE_CSV_URL}" -o "{RESPONSE_CSV_PATH}"

_check = pd.read_csv(RESPONSE_CSV_PATH, low_memory=False, skiprows=[1, 2])
print(f"parsed rows: {len(_check)} (expect ~1000)")
print(f"g_writing non-null: {_check['g_writing'].notna().sum()} (expect ~506)")

parsed rows: 1000 (expect ~1000)
g_writing non-null: 506 (expect ~506)


## 4. Build gratitude(prime) vs. routine(control) contrastive examples

In [ ]:
import numpy as np

QID46_PROMPT = (
    "Please recall carefully and in detail a specific experience in the past when you felt "
    "sincerely grateful for someone's kindness or help. Spend two minutes detailing this "
    "experience below:"
)
QID47_PROMPT = (
    "Please recall carefully and in detail the sequence of your morning routine (e.g., "
    "brushing teeth, changing clothes). Spend two minutes detailing this experience below:"
)


def load_megastudy_gratitude(csv_path):
    df = pd.read_csv(csv_path, low_memory=False, skiprows=[1, 2])
    gw = df["g_writing"].astype(str).str.strip()
    valid = gw.notna() & (gw != "") & (gw != "nan")
    df = df[valid].copy()
    df["g_writing"] = gw[valid]
    df = df[df["cond"].isin(["prime", "control"])]

    examples, labels = [], []
    for _, row in df.iterrows():
        if row["cond"] == "prime":
            user_prompt, label = QID46_PROMPT, 1
        else:
            user_prompt, label = QID47_PROMPT, 0
        examples.append({"system": "", "user": user_prompt, "assistant": row["g_writing"]})
        labels.append(label)
    return examples, np.array(labels)


examples, y = load_megastudy_gratitude(RESPONSE_CSV_PATH)
print(f"Total examples: {len(examples)}, gratitude(1)={ (y==1).sum() }, routine(0)={ (y==0).sum() }")

Total examples: 506, gratitude(1)=249, routine(0)=257


## 5. Tokenize (chat template), extract hidden states across candidate layers

In [ ]:
def to_chat_text(example):
    messages = []
    if example["system"]:
        messages.append({"role": "system", "content": example["system"]})
    messages.append({"role": "user", "content": example["user"]})
    messages.append({"role": "assistant", "content": example["assistant"]})
    return tokenizer.apply_chat_template(messages, tokenize=False)


batch_texts = [to_chat_text(ex) for ex in examples]
print(batch_texts[0][:400])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Please recall carefully and in detail the sequence of your morning routine (e.g., brushing teeth, changing clothes). Spend two minutes detailing this experience below:<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [ ]:
from torch.utils.data import DataLoader
from steer_vec_utils import TextDataset, extract_hidden_states, probe_classification

dataset = TextDataset(batch_texts)
dataloader = DataLoader(dataset, batch_size=8, shuffle=False)

CANDIDATE_LAYERS = list(range(10, 24))
LOC = 7  # hook_after_mlp_hs (residual stream after MLP)

all_hidden_states = extract_hidden_states(
    dataloader,
    tokenizer,
    model,
    assistant_tag=ASSISTANT_TAG,
    extraction_layers=CANDIDATE_LAYERS,
    extraction_locs=[LOC],
    extraction_tokens="all",  # full prompt+response, matches the real conflated manipulation
    avg_token_dim=True,
    do_final_cat=True,
)
print(all_hidden_states.shape)

  0%|          | 0/64 [00:00<?, ?it/s]

torch.Size([506, 14, 1, 1, 4096])


## 6. Compute MeanDiff + Probe vectors per layer, pick the best layer

In [ ]:
y_tensor = torch.tensor(y)
results = {}

for i, layer in enumerate(CANDIDATE_LAYERS):
    layer_hs = all_hidden_states[:, i, 0, 0, :].float()  # bfloat16 -> float32 for sklearn

    pos_hs = layer_hs[y == 1]
    neg_hs = layer_hs[y == 0]
    meandiff_vec = pos_hs.mean(dim=0) - neg_hs.mean(dim=0)

    probe_res = probe_classification(layer_hs, y_tensor, fit_intercept=False)

    results[layer] = {
        "meandiff_vec": meandiff_vec,
        "probe_weights": torch.tensor(probe_res["weights"]).flatten().float(),
        "test_acc": float(probe_res["accuracy_test"]),
    }
    print(f"layer {layer}: probe held-out test accuracy = {results[layer]['test_acc']:.3f}")

max_acc = max(results[l]["test_acc"] for l in results)
tied_layers = sorted([l for l in results if results[l]["test_acc"] == max_acc])
BEST_LAYER = tied_layers[len(tied_layers) // 2]  # among ties, pick the middle layer
print(f"Layers tied at max accuracy ({max_acc:.3f}): {tied_layers}")
print(f"BEST_LAYER (middle of tied group): {BEST_LAYER}")

Train size: 404, Test size: 102
Best C: 100 ------------------
layer 10: probe held-out test accuracy = 1.000
Train size: 404, Test size: 102
Best C: 100 ------------------
layer 11: probe held-out test accuracy = 1.000
Train size: 404, Test size: 102
Best C: 100 ------------------
layer 12: probe held-out test accuracy = 1.000
Train size: 404, Test size: 102
Best C: 100 ------------------
layer 13: probe held-out test accuracy = 1.000
Train size: 404, Test size: 102
Best C: 100 ------------------
layer 14: probe held-out test accuracy = 1.000
Train size: 404, Test size: 102
Best C: 100 ------------------
layer 15: probe held-out test accuracy = 1.000
Train size: 404, Test size: 102
Best C: 100 ------------------
layer 16: probe held-out test accuracy = 1.000
Train size: 404, Test size: 102
Best C: 100 ------------------
layer 17: probe held-out test accuracy = 1.000
Train size: 404, Test size: 102
Best C: 100 ------------------
layer 18: probe held-out test accuracy = 1.000
Train size

In [ ]:
INJECT_LAYERS = [BEST_LAYER - 1, BEST_LAYER, BEST_LAYER + 1]
INJECT_LAYERS = [l for l in INJECT_LAYERS if l in results]  # 防止超出候选层范围(10-23)

LAYER_VECTORS = {
    layer: (results[layer]["meandiff_vec"] / results[layer]["meandiff_vec"].norm()).to(model.device, dtype=torch.bfloat16)
    for layer in INJECT_LAYERS
}
print(f"注入层: {INJECT_LAYERS}")

注入层: [16, 17, 18]


In [ ]:
# The vector used for injection (normalized). VECTOR_METHOD: "meandiff" or "probe".
VECTOR_METHOD = "meandiff"
INJECT_LAYER = BEST_LAYER

raw_vec = results[INJECT_LAYER]["meandiff_vec" if VECTOR_METHOD == "meandiff" else "probe_weights"]
GRATITUDE_VECTOR = (raw_vec / raw_vec.norm()).to(model.device, dtype=torch.bfloat16)
print("GRATITUDE_VECTOR ready. layer:", INJECT_LAYER, "norm:", GRATITUDE_VECTOR.norm().item())

GRATITUDE_VECTOR ready. layer: 17 norm: 1.0


# PHASE 2: Simulation arm vs. Injection arm
## 7. Real materials (verified against the Mega Study QSF / GitHub source)

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are an AI assistant. Your task is to answer the 'New Survey Question' as if you "
    "are the person described in the 'Persona Profile' (which consists of their past survey "
    "responses). \nRemain consistent with the persona's previous answers and stated traits. "
    "Simulate their responses to new questions while accounting for human cognitive "
    "limitations, uncertainty, and biases.\nFollow all instructions provided for the new "
    "question carefully regarding the format of your answer."
)

COMBINED_PROMPT_HEADER = "## Persona Profile (This individual's past survey responses):\n"
COMBINED_PROMPT_SEPARATOR = (
    "\n\n---\n## New Survey Question & Instructions "
    "(Please respond as the persona described above):\n"
)
GENERAL_LLM_INSTRUCTION = (
    "Please answer the following questions as if you were taking this survey. The expected "
    "output is a JSON object and the format is provided in the end. \n---\n\n"
)

Q_WRITING = "{instruction}\nQuestion Type: Text Entry\nAnswer: [Masked]"

Q_MANIP_CHECK = (
    "Please indicate the intensity with which you feel each of the following emotions right now.\n"
    "Question Type: Matrix\n"
    "Options:\n  1 = Not at all\n  2 = 2\n  3 = 3\n  4 = 4\n  5 = 5\n  6 = 6\n  7 = Extremely\n"
    "1. Grateful\nAnswer: [Masked]\n"
    "2. Thankful\nAnswer: [Masked]\n"
    "3. Appreciative\nAnswer: [Masked]"
)

Q_ELEVATION = (
    "Please indicate the extent to which you are feeling each of the below feelings right now.\n"
    "Question Type: Matrix\n"
    "Options:\n  1 = Not at all\n  2 = 2\n  3 = 3\n  4 = 4\n  5 = 5\n  6 = 6\n  7 = Extremely\n"
    "1. Feeling moved\nAnswer: [Masked]\n"
    "2. Feeling uplifted\nAnswer: [Masked]\n"
    "3. Feeling optimistic about humanity\nAnswer: [Masked]\n"
    "4. Experiencing a warm feeling in the chest\nAnswer: [Masked]\n"
    "5. Desiring to help others\nAnswer: [Masked]\n"
    "6. Wanting to become a better person\nAnswer: [Masked]"
)

Q_DV1 = (
    "Please indicate how much you agree or disagree with the following statements regarding "
    "the immediate effects of tariffs on vulnerable communities:\n"
    "Question Type: Matrix\n"
    "Options:\n  1 = Completely disagree\n  2 = 2\n  3 = 3\n  4 = 4\n  5 = 5\n  6 = 6\n  7 = Completely agree\n"
    "1. I am very concerned about those most vulnerable to the effects of tariffs.\nAnswer: [Masked]\n"
    "2. I feel compassion for those most affected by rising prices or job losses due to tariffs.\nAnswer: [Masked]"
)

## 8. Structured JSON output via `outlines` (schema-guaranteed, no retry-loop needed)

In [ ]:
import json
import re
import anthropic
from google.colab import userdata

claude_client = anthropic.Anthropic(api_key=userdata.get('CLAUDE_API_KEY'))

PARSE_INSTRUCTION_TEMPLATE = """You will be shown a response from another AI model that was asked to answer survey questions using a 1-7 rating scale.

Q2 (3 items, each rated 1-7): Grateful, Thankful, Appreciative
Q3 (6 items, each rated 1-7): Feeling moved, Feeling uplifted, Feeling optimistic about humanity, Experiencing a warm feeling in the chest, Desiring to help others, Wanting to become a better person
Q4 (2 items, each rated 1-7): concern about tariffs' effect on vulnerable communities, compassion for those affected by tariffs
{q1_line}

Read the response below and extract the answers. If it clearly answers every item, output ONLY this JSON (no other text):
{{"Q2": [n, n, n], "Q3": [n, n, n, n, n, n], "Q4": [n, n]{q1_key}}}

If any item is missing, unclear, or not actually answered, output exactly:
{{"failed": true}}

Response to parse:
---
{raw_text}
---
"""


def parse_with_claude(raw_text, need_q1=False):
    q1_line = "Q1 (free text): a short first-person written response" if need_q1 else ""
    q1_key = ', "Q1": "..."' if need_q1 else ""
    prompt = PARSE_INSTRUCTION_TEMPLATE.format(q1_line=q1_line, raw_text=raw_text, q1_key=q1_key)

    resp = claude_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    claude_text = resp.content[0].text.strip()
    try:
        parsed = json.loads(claude_text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", claude_text, re.DOTALL)
        if not match:
            return claude_text, None
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            return claude_text, None

    if parsed.get("failed"):
        return claude_text, None

    # 明确校验每个字段的长度对不对，长度不对也算失败
    expected_lengths = {"Q2": 3, "Q3": 6, "Q4": 2}
    for key, length in expected_lengths.items():
        if key not in parsed or not isinstance(parsed[key], list) or len(parsed[key]) != length:
            return claude_text, None
    if need_q1 and "Q1" not in parsed:
        return claude_text, None

    return claude_text, parsed




def generate_and_extract_logged(prompt_text, pid, arm, beta, need_q1=False, max_retries=2, max_new_tokens=1500):
    """Returns a list of attempt-log rows (one per generation attempt, success or not)."""
    logs = []
    for attempt in range(1, max_retries + 1):
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
        raw_text = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        claude_raw, parsed = parse_with_claude(raw_text, need_q1=need_q1)

        logs.append({
            "pid": pid,
            "arm": arm,
            "beta": beta,
            "attempt": attempt,
            "llama_raw": raw_text,
            "claude_raw": claude_raw,
            "success": parsed is not None,
            "manip_check": parsed["Q2"] if parsed else None,
            "elevation": parsed["Q3"] if parsed else None,
            "dv1": parsed["Q4"] if parsed else None,
            "q1_writing": parsed.get("Q1") if (parsed and need_q1) else None,
        })
        if parsed is not None:
            break  # 成功了就不用再重试

    logs[-1]["is_final"] = True
    for row in logs[:-1]:
        row["is_final"] = False
    return logs

## 9. Steering hook (registered on the exact PsySET hook point, plain PyTorch API so it reliably fires on every generation step)

In [ ]:
class SteeringContext:
    """layer_vectors: {layer_idx: 该层自己的归一化向量}. 支持同时在多个层上挂钩子，
    每层各自用该层自己提取出来的方向，不是把一层的向量硬塞到别的层上。"""

    def __init__(self, model, layer_vectors, coeff):
        self.handles = []
        self.model = model
        self.layer_vectors = layer_vectors
        self.coeff = coeff

    def _make_hook(self, vector):
        def hook_fn(module, inputs, output):
            return output + self.coeff * vector
        return hook_fn

    def __enter__(self):
        for layer_idx, vector in self.layer_vectors.items():
            module = self.model.model.layers[layer_idx].hook_after_mlp_hs
            handle = module.register_forward_hook(self._make_hook(vector))
            self.handles.append(handle)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        for handle in self.handles:
            handle.remove()

## 10. Fetch personas live from the Twin-2K-500 dataset

In [ ]:
import requests


def fetch_personas(n=5, offset=0):
    url = (
        "https://datasets-server.huggingface.co/rows"
        f"?dataset=LLM-Digital-Twin%2FTwin-2K-500&config=full_persona&split=data"
        f"&offset={offset}&length={n}"
    )
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    rows = resp.json()["rows"]
    return [(r["row"]["pid"], r["row"]["persona_summary"]) for r in rows]

## 11. Simulation arm (real prompt, no injection) and Injection arm (skip Q1, steer)

In [ ]:
def build_downstream_block():
    return "Q2:\n" + Q_MANIP_CHECK + "\n\nQ3:\n" + Q_ELEVATION + "\n\nQ4:\n" + Q_DV1


def run_simulation_arm_logged(pid, persona_text, cond):
    instruction = QID46_PROMPT if cond == "prime" else QID47_PROMPT
    questions_block = "Q1:\n" + Q_WRITING.format(instruction=instruction) + "\n\n" + build_downstream_block()
    user_content = (
        COMBINED_PROMPT_HEADER + persona_text + COMBINED_PROMPT_SEPARATOR
        + GENERAL_LLM_INSTRUCTION + questions_block
    )
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_content},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    arm_name = "real_prime" if cond == "prime" else "real_control"
    return generate_and_extract_logged(prompt_text, pid=pid, arm=arm_name, beta=None, need_q1=True)


def run_injection_arm_logged(pid, persona_text, beta):
    questions_block = build_downstream_block()
    user_content = (
        COMBINED_PROMPT_HEADER + persona_text + COMBINED_PROMPT_SEPARATOR
        + GENERAL_LLM_INSTRUCTION + questions_block
    )
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_content},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    arm_name = "inject_gratitude" if beta > 0 else "inject_routine"

    with SteeringContext(model, LAYER_VECTORS, beta):
        logs = generate_and_extract_logged(prompt_text, pid=pid, arm=arm_name, beta=abs(beta), need_q1=False)
    return logs

## 12. Calibrate β on a small subset (match Arm-1's manipulation-check average), then run the full 4-arm comparison
`BETA` is fixed after calibration and reused as-is for Elevation / DV1 (do not re-tune per measure).

In [ ]:
BETA_RANGE = [2.0,4.5]
N_CALIB_PERSONAS = 3

calib_personas = fetch_personas(n=N_CALIB_PERSONAS, offset=0)

for beta_try in BETA_RANGE:
    scores = []
    failed = False
    for pid, persona_text in calib_personas:
        logs = run_injection_arm_logged(pid, persona_text, beta=beta_try)
        final = logs[-1]
        if final["success"]:
            scores.append(sum(final["manip_check"]) / 3)
        else:
            failed = True
            break

    if scores:
        if failed:
            print(f"beta={beta_try}: 前 {len(scores)} 个persona成功, mean manip_check = {sum(scores)/len(scores):.2f}, 但随后出现失败")
        else:
            print(f"beta={beta_try}: mean manip_check = {sum(scores)/len(scores):.2f} (n={len(scores)}, target: ~6)")
    else:
        print(f"beta={beta_try}: 第一个persona就失败了")

    if failed:
        print(f"beta={beta_try}: 出现失败, 停止整个β扫描")
        break

    # quick text-quality eyeball check
    pid0, persona_text0 = calib_personas[0]
    questions_block = build_downstream_block()
    user_content = (
        COMBINED_PROMPT_HEADER + persona_text0 + COMBINED_PROMPT_SEPARATOR
        + GENERAL_LLM_INSTRUCTION + questions_block
    )
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_content},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with SteeringContext(model, LAYER_VECTORS, beta_try):
        with torch.no_grad():
            output_ids = model.generate(
                inputs.input_ids, attention_mask=inputs.attention_mask,
                max_new_tokens=300, do_sample=True, temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
    sample_text = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"  [text quality check @ beta={beta_try}]: {sample_text[:200]!r}")
    print()

beta=2.0: mean manip_check = 4.44 (n=3, target: ~6)
  [text quality check @ beta=2.0]: "I'll do my best to answer this survey question as the person described above. Here are my responses:\n\n**Q2:**\n\n1. Grateful: 6\n2. Thankful: 5\n3. Appreciative: 4\n\nI'm feeling grateful and thankful becau"

beta=4.5: 前 1 个persona成功, mean manip_check = 7.00, 但随后出现失败
beta=4.5: 出现失败, 停止整个β扫描


In [ ]:
from tqdm.auto import tqdm

N_PERSONAS = 100
BETA_VALUES_GRATITUDE = [0.0, 1.8, 2.5, 3.2, 3.9]
BETA_VALUES_ROUTINE = [2.5]

personas = fetch_personas(n=N_PERSONAS, offset=0)
all_logs = []

total_combos_per_persona = 2 + len(BETA_VALUES_GRATITUDE) + len(BETA_VALUES_ROUTINE)
combo_count = 0

for i, (pid, persona_text) in enumerate(tqdm(personas, desc="Personas")):
    print(f"\n--- persona {i+1}/{len(personas)} (pid={pid}) ---")

    logs = run_simulation_arm_logged(pid, persona_text, cond="prime")
    all_logs.extend(logs)
    combo_count += 1
    print(f"  [{combo_count}] real_prime done (success={logs[-1]['success']}, attempts={logs[-1]['attempt']})")

    logs = run_simulation_arm_logged(pid, persona_text, cond="control")
    all_logs.extend(logs)
    combo_count += 1
    print(f"  [{combo_count}] real_control done (success={logs[-1]['success']}, attempts={logs[-1]['attempt']})")

    for beta in BETA_VALUES_GRATITUDE:
        logs = run_injection_arm_logged(pid, persona_text, beta=+beta)
        all_logs.extend(logs)
        combo_count += 1
        print(f"  [{combo_count}] inject_gratitude beta={beta} done (success={logs[-1]['success']}, attempts={logs[-1]['attempt']})")

    for beta in BETA_VALUES_ROUTINE:
        logs = run_injection_arm_logged(pid, persona_text, beta=-beta)
        all_logs.extend(logs)
        combo_count += 1
        print(f"  [{combo_count}] inject_routine beta={beta} done (success={logs[-1]['success']}, attempts={logs[-1]['attempt']})")

    print(f"pid={pid} done ({combo_count}/{total_combos_per_persona * len(personas)} 个组合已完成)")

results_df = pd.DataFrame(all_logs)
results_df.to_csv("affective_priming_full_log.csv", index=False)
results_df


Personas:   0%|          | 0/100 [00:00<?, ?it/s]


--- persona 1/100 (pid=574) ---
  [1] real_prime done (success=True, attempts=1)
  [2] real_control done (success=True, attempts=1)
  [3] inject_gratitude beta=0.0 done (success=True, attempts=1)
  [4] inject_gratitude beta=1.8 done (success=True, attempts=1)
  [5] inject_gratitude beta=2.5 done (success=True, attempts=2)
  [6] inject_gratitude beta=3.2 done (success=True, attempts=1)
  [7] inject_gratitude beta=3.9 done (success=True, attempts=1)
  [8] inject_routine beta=2.5 done (success=True, attempts=1)
pid=574 done (8/800 个组合已完成)

--- persona 2/100 (pid=2001) ---
  [9] real_prime done (success=True, attempts=1)
  [10] real_control done (success=True, attempts=1)
  [11] inject_gratitude beta=0.0 done (success=True, attempts=1)
  [12] inject_gratitude beta=1.8 done (success=True, attempts=1)
  [13] inject_gratitude beta=2.5 done (success=True, attempts=1)
  [14] inject_gratitude beta=3.2 done (success=True, attempts=1)
  [15] inject_gratitude beta=3.9 done (success=True, attempts=

,pid,arm,beta,attempt,llama_raw,claude_raw,success,manip_check,elevation,dv1,q1_writing,is_final
0,574,real_prime,NaN,1,Here are the answers to the survey questions a...,"```json\n{""Q2"": [4, 5, 3], ""Q3"": [2, 3, 2, 1, ...",True,"[4, 5, 3]","[2, 3, 2, 1, 4, 3]","[5, 6]",I recall a specific experience in the past whe...,True
1,574,real_control,NaN,1,Here are my responses to the new survey questi...,"```json\n{""Q2"": [4, 5, 4], ""Q3"": [3, 4, 5, 4, ...",True,"[4, 5, 4]","[3, 4, 5, 4, 5, 6]","[6, 7]",My morning routine typically starts at 6:00 AM...,True
2,574,inject_routine,0.0,1,Based on my previous answers and the persona p...,"{""Q2"": [4, 5, 4], ""Q3"": [3, 4, 5, 4, 6, 6], ""Q...",True,"[4, 5, 4]","[3, 4, 5, 4, 6, 6]","[6, 6]",None,True
3,574,inject_gratitude,1.8,1,I'm happy to help and answer these questions a...,"{""Q2"": [4, 5, 3], ""Q3"": [3, 4, 5, 2, 5, 4], ""Q...",True,"[4, 5, 3]","[3, 4, 5, 2, 5, 4]","[6, 6]",None,True
4,574,inject_gratitude,2.5,1,I'll respond to these questions as the person ...,"{""failed"": true}",False,None,None,None,None,False
...,...,...,...,...,...,...,...,...,...,...,...,...
875,1521,inject_gratitude,2.5,1,Here are the answers from the perspective of t...,"{""Q2"": [7, 5, 6], ""Q3"": [6, 5, 6, 7, 6, 5], ""Q...",True,"[7, 5, 6]","[6, 5, 6, 7, 6, 5]","[7, 7]",None,True
876,1521,inject_gratitude,3.2,1,I was touched by the kindness and generosity o...,"```json\n{""Q2"": [7, 7, 7], ""Q3"": [6, 6, 6, 6, ...",True,"[7, 7, 7]","[6, 6, 6, 6, 6, 6]","[7, 7]",None,True
877,1521,inject_gratitude,3.9,1,I was truly grateful for the help and support ...,"{""failed"": true}",False,None,None,None,None,False
878,1521,inject_gratitude,3.9,2,I was really touched by the kindness of strang...,"{""failed"": true}",False,None,None,None,None,True


## 13. Summary: mean scores per arm

In [1]:
def mean_of_list_col(series):
    return series.apply(lambda x: sum(x) / len(x)).mean()


final_df = results_df[(results_df["is_final"]) & (results_df["success"])]

summary = final_df.groupby(["arm", "beta"], dropna=False).agg(
    n=("pid", "count"),
    manip_check_mean=("manip_check", mean_of_list_col),
    elevation_mean=("elevation", mean_of_list_col),
    dv1_mean=("dv1", mean_of_list_col),
)
print(summary)

# 顺带看一下每个组合的失败率，现在数据够细可以直接算出来
fail_rate = results_df[results_df["is_final"]].groupby(["arm", "beta"], dropna=False)["success"].agg(
    n_total="count", n_success="sum"
)
fail_rate["fail_rate"] = 1 - fail_rate["n_success"] / fail_rate["n_total"]
print()
print(fail_rate)

from google.colab import files
files.download("affective_priming_full_log.csv")

NameError: name 'results_df' is not defined